# UCA Direction Finding Using I/Q Phase Correlation

This notebook develops the direction-finding simulation from the received I/Q samples.

The relative phases between neighbouring antenna signals are extracted from the received I/Q data and compared with pre-calculated phase signatures to estimate the transmitter azimuth.

In [ ]:
using FFTW
using LinearAlgebra
using Plots
using Random
plotly(); # plotyly backend for Plots

In [ ]:
j = im

# Speed of light
c = 3e8

# Receiver sampling frequency
fs = 20e6
dt = 1 / fs

# Number of samples
Nsamples = 2^15

# Duration of one antenna acquisition
Tcapture = Nsamples / fs

# Time vector
t = (0:Nsamples - 1) * dt
t_max = Nsamples * dt

# Reciver centre frequency and transmitted frequency
fc = 433.0e6      # Receiver centre frequency = 433.0 MHz
f0 = fc + 100e3

# Wavelength of the transmitted signal
λ = c / f0;

#@show Tcapture

### Reference and Switched Antenna Acquisition

Antenna 1 is continuously connected to RX1 and acts as the reference.

Antennas 2 to 6 are switched sequentially into RX2. During each switch state,
RX1 and RX2 are sampled simultaneously, so the acquisition-time phase is common
to both channels and cancels in the relative phase measurement.

Thus, for each switched antenna $n$,

$$
\Delta\phi_{1n}
=
\angle(V_1V_n^*).
$$

In [ ]:
# Number of antennas on the circle
Nant = 6

# Antenna 1 is permanently connected to RX1
# Antenna 2 to 6 are switched sequentially into RX2
Nswitch = Nant - 1

# Antennas connected through the RF switch
switched_antennas = 2:Nant

# Start time of each switch state
t_state = zeros(Nswitch)

for state = 1:Nswitch
    t_state[state] = (state - 1) * Tcapture
end

# Circle radius
Rcircle = 0.4 * λ

# Store the antenna positions
A = [zeros(3) for n in 1:Nant]

for n = 1:Nant
    
    theta = (2π / Nant) * (n - 1)
    
    x = Rcircle * cos(theta)
    y = Rcircle * sin(theta)
    z = 0
    
    A[n] = [x, y, z]
    
end
#@show switched_antennas
#@show t_state

In [ ]:
# True source azimuth and distance used only to generate the simulation
phi0 = deg2rad(30)
Rsource = 2

# Reflected-path direction
theta_reflection = 100
phi_reflection = deg2rad(theta_reflection)

# Source position
P = [Rsource * cos(phi0), Rsource * sin(phi0), 0]

# Virtual reflected source position
P_reflection = [Rsource * cos(phi_reflection), Rsource * sin(phi_reflection), 0]

# Store distance and propagation delay for each antenna
range = zeros(Nant)
tau = zeros(Nant)

# Store reflected-path distance and delay
range_reflection = zeros(Nant)
tau_reflection = zeros(Nant)

for n = 1:Nant
    
    # Direct path
    R = A[n] - P
    
    range[n] = norm(R)
    tau[n] = range[n] / c
    
    # Reflected path
    R_reflection = A[n] - P_reflection
    
    range_reflection[n] = norm(R_reflection)
    tau_reflection[n] = range_reflection[n] / c
end

#@show range
#@show tau
#@show range_reflection
#@show tau_reflection

## 3. I/Q Data at Each Antenna

The signal transmitted at RF frequency $f_0$ is delayed by the propagation time to each antenna.

For antenna $n$,

$$
\tau_n = \frac{r_n}{c}
$$

and the received RF signal is

$$
x_n(t)
=
A e^{j2\pi f_0(t-\tau_n)}.
$$

Expanding the delay term,

$$
x_n(t)
=
A e^{-j2\pi f_0\tau_n}
e^{j2\pi f_0t}.
$$

The receiver is centred at frequency $f_c$. After downconversion, the
received I/Q signal becomes

$$
v_n(t)
=
A e^{-j2\pi f_0\tau_n}
e^{j2\pi(f_0-f_c)t}.
$$

The term

$$
e^{-j2\pi f_0\tau_n}
$$

contains the propagation phase caused by the distance from the source
to antenna $n$.

The baseband frequency is

$$
f_{\mathrm{BB}} = f_0-f_c.
$$


For each switch state, antenna 1 and the selected antenna are sampled
simultaneously. The same baseband signal model therefore applies to both
receive channels, with the propagation phase determined by the distance from
the source to the corresponding antenna.

### Additive Gaussian Noise

Receiver noise is modelled as additive complex Gaussian noise

$$
n[k] = n_I[k] + jn_Q[k],
$$

where

$$
n_I,n_Q \sim \mathcal{N}(0,\sigma^2).
$$

For the complex I/Q signal, the signal-to-noise ratio is

$$
\mathrm{SNR}
=
\frac{P_s}{P_n}.
$$

Since the signal magnitude is $A$,

$$
P_s=A^2,
$$

while the complex noise power is

$$
P_n=2\sigma^2.
$$

Therefore,

$$
\boxed{
\sigma
=
\sqrt{
\frac{A^2}
{(2)*\,10^{\mathrm{SNR}_{dB}/10}}
}
}
$$

The noise is added directly to the received I/Q samples before FFT processing.

In [ ]:
# Signal amplitude
Amp = 1

# Reflected-path relative amplitude
α = 0.8

# Baseband frequency
fBB = f0 - fc

# Signal-to-noise ratio
SNR_dB = -20

# Noise standard deviation
σ = sqrt(Amp^2 / (2 * 10^(SNR_dB / 10)))
# σ = 0

# Store I/Q data for RX1 and RX2 for each switch state
IQ_RX1 = [zeros(ComplexF64, Nsamples) for state in 1:Nswitch]
IQ_RX2 = [zeros(ComplexF64, Nsamples) for state in 1:Nswitch]

for state = 1:Nswitch
    
    Random.seed!(1234)
    
    # Antenna currently connected to RX2
    n = switched_antennas[state]
    
    # Both receivers are sampled at the same time
    t_current = t .+ t_state[state]
    
    
    # RX1: Fixed reference antenna 1
    # ------------------------------
    
    # Phase shift caused by propagation to antenna 1
    phase_directSig_RX1 = exp(-j * 2π * f0 * tau[1])
    
    # Received signal
    directSig_RX1 = Amp * phase_directSig_RX1 .* exp.(j * 2π * fBB .* t_current)
    
    # Reflected path
    phase_reflection_RX1 = exp(-j * 2π * f0 * tau_reflection[1])
    
    reflected_signal_RX1 = α * Amp * phase_reflection_RX1 .* exp.(j * 2π * fBB .* t_current)
    
    # Complex Gaussian noise
    noise_RX1 = σ .* randn(Nsamples) .+ j .* σ .* randn(Nsamples)
    
    # I/Q signal received at antenna 1
    IQ_RX1[state] = directSig_RX1 + reflected_signal_RX1 + noise_RX1
    
    
    # RX2: Switched antenna
    # ------------------------------
    
    # Phase shift caused by propagation to antenna n
    phase_directSig_RX2 = exp(-j * 2π * f0 * tau[n])
    
    # Received signal
    directSig_RX2 = Amp * phase_directSig_RX2 .* exp.(j * 2π * fBB .* t_current)
    
    # Reflected path
    phase_reflection_RX2 = exp(-j * 2π * f0 * tau_reflection[n])
    
    reflected_signal_RX2 = α * Amp * phase_reflection_RX2 .* exp.(j * 2π * fBB .* t_current)
    
    # Complex Gaussian noise
    noise_RX2 = σ .* randn(Nsamples) .+ j .* σ .* randn(Nsamples)
    
    # I/Q signal received at antenna n
    IQ_RX2[state] = directSig_RX2 + reflected_signal_RX2 + noise_RX2
    
end
#@show SNR_dB
#@show σ

In [ ]:
# Complex Gaussian noise visualisation

noise_plot = σ .* randn(Nsamples) .+ j .* σ .* randn(Nsamples)
histogram(
    real.(noise_plot), 
    bin = 60, 
    xlabel = "Noise Amplitude", 
    ylabel = "Number of Samples", 
    title = "Gaussian Noise - I Cmponent",
    label = false
)

In [ ]:
# Store the FFT for each switch state
FFT_RX1 = [zeros(ComplexF64, Nsamples) for n in 1:Nswitch]
FFT_RX2 = [zeros(ComplexF64, Nsamples) for n in 1:Nswitch]

# Store the complex FFT value at the signal frequency
FFT_peak_RX1 = zeros(ComplexF64, Nswitch)
FFT_peak_RX2 = zeros(ComplexF64, Nswitch)

# Store the FFT-bin index used for each switch state
peak_index = zeros(Int, Nswitch)


for state = 1:Nswitch
    
    # FFT of the two simultaneously received signals
    FFT_RX1[state] = fft(IQ_RX1[state])
    FFT_RX2[state] = fft(IQ_RX2[state])
    
    # Find the signal peak using the reference channel
    peak_index[state] = argmax(abs.(FFT_RX1[state]))
        
    # Use the same FFT bin for RX1 and RX2
    FFT_peak_RX1[state] = FFT_RX1[state][peak_index[state]]
    FFT_peak_RX2[state] = FFT_RX2[state][peak_index[state]] 
    
end

#@show peak_index

In [ ]:
# FFT frequency resolution
df = fs / Nsamples

# Frequency axis
frequency_axis = (0:Nsamples-1) .* df

# Antenna 1 is permanently connected to RX1
p = plot(
    frequency_axis,
    abs.(FFT_RX1[1]),
    xlabel = "Frequency (Hz)",
    ylabel = "FFT Magnitude",
    title = "FFT Spectrum of All Antennas",
    label = "Antenna 1",
    xlims = (0, 300e3)
)

# Antennas 2 to 6 are measured through RX2
for state = 1:Nswitch

    antenna_number = switched_antennas[state]

    plot!(
        p,
        frequency_axis,
        abs.(FFT_RX2[state]),
        label = "Antenna $antenna_number"
    )

end

display(p)

In [ ]:
# Store the relative complex FFT values:
# A1-A2, A1-A3, A1-A4, A1-A5, A1-A6
relative_FFT = zeros(ComplexF64, Nswitch)

for state = 1:Nswitch

    V1 = FFT_peak_RX1[state]
    Vn = FFT_peak_RX2[state]

    # Relative complex value between antenna 1 and antenna n
    relative_FFT[state] = V1 * conj(Vn)

end


# Normalise so that only the relative phase is shown
relative_FFT_normalised =
    relative_FFT ./ abs.(relative_FFT)


# Plot the relative phase phasors
p = scatter(
    real.(relative_FFT_normalised),
    imag.(relative_FFT_normalised),
    xlabel = "Real",
    ylabel = "Imaginary",
    title = "Relative FFT Phase Values",
    label = false,
    aspect_ratio = :equal,
    xlims = (-1.2, 1.2),
    ylims = (-1.2, 1.2)
)


# Label each antenna pair
for state = 1:Nswitch

    antenna_number = switched_antennas[state]

    annotate!(
        p,
        real(relative_FFT_normalised[state]),
        imag(relative_FFT_normalised[state]),
        text("A1-A$antenna_number", 9)
    )

end

display(p)

In [ ]:
# FFT phase meaasured on RX1 and RX2 for each switch state
FFT_phase_RX1_deg = zeros(Nswitch)
FFT_phase_RX2_deg = zeros(Nswitch)

for state = 1:Nswitch
    
    FFT_phase_RX1_deg[state] = angle(FFT_peak_RX1[state]) * 180 / π
    
    FFT_phase_RX2_deg[state] = angle(FFT_peak_RX2[state]) * 180 / π
    
end

#@show FFT_phase_RX1_deg
#@show FFT_phase_RX2_deg

## 4. Relative Phase Between Antennas

The received baseband I/Q signal at antenna $n$ is

$$
v_n(t)
=
A e^{-j2\pi f_0\tau_n}
e^{j2\pi f_{BB}t},
$$

where

$$
f_{BB}=f_0-f_c.
$$

The propagation phase at antenna $n$ is

$$
\phi_n
=
-2\pi f_0\tau_n
=
-\frac{2\pi r_n}{\lambda}.
$$

The sampled signal is therefore

$$
v_n[k]
=
A e^{j\phi_n}
e^{j2\pi f_{BB}k/f_s}.
$$

The FFT of the received I/Q samples is

$$
V_n[m]
=
\sum_{k=0}^{N-1}
v_n[k]e^{-j2\pi mk/N}.
$$

The signal frequency is identified from the FFT peak

$$
m_{\text{peak}}
=
\underset{m}{\operatorname{arg\,max}}
|V_n[m]|.
$$

The complex FFT value at this bin is

$$
V_n
=
V_n[m_{\text{peak}}]
=
B_n e^{j\psi_n}.
$$

Antenna 1 is permanently connected to RX1 and acts as the reference antenna.
Antennas 2 to 6 are switched sequentially into RX2.

For each switch state, antenna 1 and the selected antenna $n$ are sampled
simultaneously. Their relative phase is therefore obtained from

$$
V_1V_n^*
=
B_1B_n e^{j(\psi_1-\psi_n)},
$$

so that

$$
\boxed{
\Delta\phi_{1n}
=
\angle(V_1V_n^*)
}
$$

for

$$
n=2,3,4,5,6.
$$

The six-element circular array therefore produces the five-element measured
phase signature

$$
\boxed{
\mathbf{p}_{\text{measured}}
=
\begin{bmatrix}
\Delta\phi_{12} &
\Delta\phi_{13} &
\Delta\phi_{14} &
\Delta\phi_{15} &
\Delta\phi_{16}
\end{bmatrix}.
}
$$

This phase signature is compared with the expected phase signature for each
candidate azimuth to estimate the source direction.

In [ ]:
# Relative phase measurements:
# Δφ12, Δφ13, Δφ14, Δφ15, Δφ16
phase_difference_deg = zeros(Nswitch)

for state = 1:Nswitch

    # FFT peak from antenna 1  on RX1
    V1 = FFT_peak_RX1[state]
    
    # FFT peak from the switched antenna on RX2
    Vn = FFT_peak_RX2[state]
    
    # Relative phase: ϕ1 - ϕn
    phase_difference_deg[state] = angle(V1 * conj(Vn)) * 180 / π
    
end

#@show phase_difference_deg

### Far-Field Phase Lookup

The received I/Q signals are generated using the exact source-to-antenna
distances

$$
r_n = \|\mathbf{A}_n-\mathbf{P}\|.
$$

The phase lookup table assumes that the transmitter is sufficiently far from
the array for the incoming wavefront to be approximated as planar. The
path-length difference between neighbouring antennas is then

$$
r_n-r_m
\approx
-\hat{\mathbf{u}}\cdot(\mathbf{A}_n-\mathbf{A}_m),
$$

giving the expected phase difference

$$
\Delta\phi_{nm}
\approx
\frac{2\pi}{\lambda}
\hat{\mathbf{u}}\cdot(\mathbf{A}_n-\mathbf{A}_m).
$$

In [ ]:
# Azimuth angles to pre-calculate
azimuth_scan_deg = 0:359

# Store the RMSE for each candidate angle
RMSE_table = zeros(length(azimuth_scan_deg))

for angle_index = 1:length(azimuth_scan_deg)
    
    theta_deg = azimuth_scan_deg[angle_index]
    theta = deg2rad(theta_deg)

    # Unit vector pointing towards the candidate source direction
    u = [cos(theta), sin(theta), 0]
    
    # Expected phase differences:
    # Δφ12, Δφ13, Δφ14, Δφ15, Δφ16
    expected_phase_deg = zeros(Nswitch)

    for state = 1:Nswitch
        n = switched_antennas[state]
    
        # Far-field path difference between antenna 1 and antenna n
        path_difference = dot(u, A[1] - A[n])
    
        # Convert path difference to phase difference
        expected_phase_deg[state] = (2π / λ) * path_difference * 180 / π
    
        # Wrap to -180 to 180 degrees
        expected_phase_deg[state] = angle(exp(j * expected_phase_deg[state] * π / 180)) * 180 / π
    end
    
    # Difference between measured and expected phase signatures
    difference = phase_difference_deg - expected_phase_deg

    # Wrap the phase error
    wrapped_difference = angle.(exp.(j .* difference .* π / 180)) .* π

    # RMSE for this candidate angle
    RMSE_table[angle_index] = sqrt(sum(wrapped_difference.^2) / Nswitch)
end

# Best-matching azimuth
minimum_RMSE, minimum_index = findmin(RMSE_table)

estimated_azimuth = azimuth_scan_deg[minimum_index]

@show minimum_RMSE
@show estimated_azimuth

In [ ]:
plot(
    azimuth_scan_deg,
    RMSE_table,
    xlabel = "Azimuth (degrees)",
    ylabel = "RMSE (degrees)",
    title = "RMSE versus Candidate Azimuth",
    legend = false
)

## Simulation Test Summary

### Baseline

| Variable changed | Value | Estimated azimuth | Minimum RMSE |
|---|---:|---:|---:|
| — | Basic estimator | $30^\circ$ | $0.051^\circ$ |

### Source Range Test

The source distance **`Rsource`** was varied while the true azimuth remained
$\theta_0 = 30^\circ$.

| **`Rsource`** (m) | Estimated azimuth | Minimum RMSE |
|---:|---:|---:|
| 0.50 | $29^\circ$ | $0.769^\circ$ |
| 0.75 | $29^\circ$ | $0.567^\circ$ |
| 1.00 | $29^\circ$ | $0.445^\circ$ |
| 2.00 | $29^\circ$ | $0.242^\circ$ |
| 5.00 | $30^\circ$ | $0.102^\circ$ |
| 10.00 | $30^\circ$ | $0.051^\circ$ |
| 50.00 | $30^\circ$ | $0.010^\circ$ |
| 100.00 | $30^\circ$ | $0.005^\circ$ |

Using the reference phase differences
$\left[\Delta\phi_{12},\Delta\phi_{13},\Delta\phi_{14},\Delta\phi_{15},\Delta\phi_{16}\right]$
produced a much smaller phase-pattern mismatch than the earlier neighbouring phase differences
$\left[\Delta\phi_{12},\Delta\phi_{23},\Delta\phi_{34},\Delta\phi_{45},\Delta\phi_{56},\Delta\phi_{61}\right]$.
The minimum RMSE decreased from $20.89^\circ$ to $0.769^\circ$ at $0.5$ m and from $1.25^\circ$ to $0.051^\circ$ at $10$ m, corresponding to reductions of approximately $96\%$ in both cases.

### Sequential Acquisition Test

Sequential antenna acquisition was introduced using **`t_antenna`**.

| Condition | Estimated azimuth | Minimum RMSE |
|---|---:|---:|
| Switching phase not corrected | $29^\circ$ | $60.17^\circ$ |
| Switching phase corrected | $30^\circ$ | $1.221^\circ$ |

### Noise Test

Gaussian noise was added independently to RX1 and RX2 while the source remained at
$R_{\mathrm{source}} = 10$ m and $\theta_0 = 30^\circ$, with multipath disabled.

A fixed random seed, **`Random.seed!(1234)`**, was used so that each SNR value was
tested using the same underlying noise realisation, with only the noise amplitude
changing according to **`SNR_dB`**.

| **`SNR_dB`** | Estimated azimuth | Minimum RMSE |
|---:|---:|---:|
| 25 dB | $30^\circ$ | $0.0515^\circ$ |
| 10 dB | $30^\circ$ | $0.0523^\circ$ |
| 5 dB | $30^\circ$ | $0.0539^\circ$ |
| 0 dB | $30^\circ$ | $0.0588^\circ$ |
| -5 dB | $30^\circ$ | $0.0721^\circ$ |
| -10 dB | $30^\circ$ | $0.1032^\circ$ |
| -15 dB | $31^\circ$ | $0.1644^\circ$ |
| -30 dB | $33^\circ$ | $0.7386^\circ$ |
| -33 dB | $261^\circ$ | $3.1791^\circ$ |

The estimator remained close to the true $30^\circ$ direction over a wide SNR
range, with only a $1^\circ$ error appearing at $-15$ dB and a $3^\circ$ error
at $-30$ dB. At $-33$ dB the estimated direction changed to $261^\circ$,
indicating that the phase measurements had become too strongly noise-dominated
for the correct azimuth to be identified in this noise realisation.

### Multipath Test

The reflected-path amplitude **`α`** was varied while the direct source remained at
$R_{\mathrm{source}} = 10$ m and $\theta_0 = 30^\circ$. The reflected source was
positioned at $100^\circ$, while noise was disabled so that the effect of multipath
could be observed independently.

| **`α`** | Estimated azimuth | Minimum RMSE |
|---:|---:|---:|
| 0.0 | $30^\circ$ | $0.051^\circ$ |
| 0.1 | $32^\circ$ | $0.142^\circ$ |
| 0.2 | $34^\circ$ | $0.265^\circ$ |
| 0.3 | $36^\circ$ | $0.403^\circ$ |
| 0.5 | $43^\circ$ | $0.685^\circ$ |
| 0.8 | $58^\circ$ | $1.062^\circ$ |
| 1.0 | $68^\circ$ | $1.173^\circ$ |

As the reflected-path amplitude increased, the estimated azimuth was progressively
pulled away from the true $30^\circ$ direction towards the reflected signal at
$100^\circ$. At $\alpha = 0.1$ the error was only $2^\circ$, whereas an equal-strength
reflection at $\alpha = 1$ increased the error to $38^\circ$.

### Combined Stress-Case Test

Noise, multipath and source distance were varied simultaneously to test the
estimator under more realistic adverse conditions. The true source azimuth
remained $\theta_0 = 30^\circ$, with the reflected path arriving from
$100^\circ$.

| Case | **`Rsource`** (m) | **`SNR_dB`** | **`α`** | Estimated azimuth | Angular error | Minimum RMSE |
|---:|---:|---:|---:|---:|---:|---:|
| 1 | 10 | $-10$ dB | 0.3 | $28^\circ$ | $2^\circ$ | $1.319^\circ$ |
| 2 | 5 | $-15$ dB | 0.5 | $44^\circ$ | $14^\circ$ | $0.719^\circ$ |
| 3 | 2 | $-20$ dB | 0.8 | $56^\circ$ | $26^\circ$ | $1.574^\circ$ |

The estimator remained close to the true direction in Case 1, with only a
$2^\circ$ angular error despite the simultaneous presence of noise and
multipath. As the conditions became more severe, the estimate was increasingly
pulled away from the true $30^\circ$ direction, reaching errors of $14^\circ$
and $26^\circ$ in Cases 2 and 3 respectively.